In [2]:
pip install cartopy netCDF4 rasterio

  Using cached affine-2.4.0-py3-none-any.whl.metadata (4.0 kB)
  Using cached cligj-0.7.2-py3-none-any.whl.metadata (5.0 kB)
   ---------------------------------------- 0.0/11.0 MB ? eta -:--:--
   --------------- ------------------------ 4.2/11.0 MB 29.6 MB/s eta 0:00:01
   ---------------------------------------- 11.0/11.0 MB 29.7 MB/s  0:00:00
   ---------------------------------------- 0.0/21.3 MB ? eta -:--:--
   ------------------------------------ --- 19.4/21.3 MB 94.6 MB/s eta 0:00:01
   ---------------------------------------- 21.3/21.3 MB 85.1 MB/s  0:00:00
   ---------------------------------------- 0.0/30.1 MB ? eta -:--:--
   -------------------------------- ------- 24.4/30.1 MB 116.9 MB/s eta 0:00:01
   ---------------------------------------- 30.1/30.1 MB 104.1 MB/s  0:00:00
Using cached cligj-0.7.2-py3-none-any.whl (7.1 kB)
   ---------------------------------------- 0.0/6.3 MB ? eta -:--:--
   ---------------------------------------- 6.3/6.3 MB 98.7 MB/s  0:00:00
   --

In [2]:
"""
spectral_index.py
=================
Módulo de cálculo de índices espectrais a partir das bandas do satélite
GOES-16/ABI (produto ABI-L1b ou ABI-L2-CMIPF processado).

Índices implementados:
    - NDVI  : Normalized Difference Vegetation Index
    - NBR   : Normalized Burn Ratio
    - NBR2  : Normalized Burn Ratio 2
    - NDMI  : Normalized Difference Moisture Index
    - MIRBI : Mid-Infrared Burned Index
    - EVI   : Enhanced Vegetation Index
    - SAVI  : Soil-Adjusted Vegetation Index

Mapeamento de bandas ABI utilizadas:
    blue  → M6C01 (0.47 μm – Azul)
    red   → M6C02 (0.64 μm – Vermelho)
    nir   → M6C03 (0.86 μm – NIR)
    swir  → M6C05 (1.61 μm – SWIR1)
    swir2 → M6C06 (2.25 μm – SWIR2)

Estrutura de saída:
    <output_base>/<INDICE>/<ano>/<dia>/<hora_HH>/<horaHHMMSS>.tif

Dependências:
    rasterio, numpy, inspect

Uso típico:
    matches = encontrar_matches_por_data_hora('Arquivos/FINAL')
    for nome, func in INDICES.items():
        processar_indice(matches, nome, func, output_base_dir='Arquivos')
"""

import inspect
import os
import re
from collections import defaultdict
from pathlib import Path
from typing import Callable, Dict, List, Optional

import numpy as np
import rasterio

# Fallback para tqdm: se não instalado, usa iterador simples
try:
    from tqdm import tqdm
except ImportError:
    def tqdm(iterable, desc="", **kwargs):
        """Substituto simples de tqdm quando a biblioteca não está instalada."""
        print(f"{desc}..." if desc else "Processando...")
        return iterable

# ---------------------------------------------------------------------------
# Constantes do módulo
# ---------------------------------------------------------------------------

# Substrings identificadoras de cada banda no nome do arquivo GOES-16.
# 'M6C0X' é mais específico que 'C0X' e evita falsos positivos.
BAND_MAP: Dict[str, str] = {
    "blue":  "M6C01",   # 0.47 μm – Azul
    "red":   "M6C02",   # 0.64 μm – Vermelho
    "nir":   "M6C03",   # 0.86 μm – NIR
    "swir":  "M6C05",   # 1.61 μm – SWIR1
    "swir2": "M6C06",   # 2.25 μm – SWIR2
}

# Conjunto de bandas obrigatórias para que um grupo seja processado
BANDAS_NECESSARIAS = frozenset(BAND_MAP.keys())

# Padrão temporal nos nomes de arquivo GOES-16: _sYYYYDDDHHMMSS
_PADRAO_TEMPORAL = re.compile(r"_s(\d{4})(\d{3})(\d+)")


# ---------------------------------------------------------------------------
# 1. Identificação e agrupamento de arquivos por data/hora
# ---------------------------------------------------------------------------

def identificar_bandas(lista_arquivos: List[str]) -> Dict[str, str]:
    """
    Identifica as bandas espectrais presentes em uma lista de arquivos.

    Usa `BAND_MAP` para associar cada arquivo à sua banda pelo nome.
    Se dois arquivos corresponderem à mesma banda, o último é mantido.

    Parâmetros:
        lista_arquivos (list[str]): Caminhos dos arquivos a inspecionar.

    Retorna:
        dict[str, str]: Mapeamento {nome_banda: caminho_arquivo}.
    """
    bandas: Dict[str, str] = {}
    for caminho in lista_arquivos:
        nome = os.path.basename(caminho)
        for banda, substr in BAND_MAP.items():
            if substr in nome:
                bandas[banda] = caminho
                break   # cada arquivo pertence a uma única banda
    return bandas


def encontrar_matches_por_data_hora(diretorio_base: str) -> Dict[str, Dict[str, str]]:
    """
    Varre recursivamente um diretório e agrupa arquivos por chave temporal.

    Apenas grupos com todas as 5 bandas necessárias (blue, red, nir, swir,
    swir2) são incluídos no resultado. Grupos incompletos são reportados.

    Parâmetros:
        diretorio_base (str): Raiz da árvore de diretórios a varrer.

    Retorna:
        dict: {chave_temporal: {nome_banda: caminho_arquivo}}
              onde chave_temporal = '_sYYYYDDDHHMMSS'.

    Levanta:
        FileNotFoundError: Se `diretorio_base` não existir.
    """
    if not os.path.isdir(diretorio_base):
        raise FileNotFoundError(f"Diretório base não encontrado: '{diretorio_base}'")

    # Agrupa caminhos por chave temporal usando defaultdict
    grupos: Dict[str, List[str]] = defaultdict(list)
    for root, _, files in os.walk(diretorio_base):
        for nome in files:
            match = _PADRAO_TEMPORAL.search(nome)
            if match:
                chave = f"_s{match.group(1)}{match.group(2)}{match.group(3)}"
                grupos[chave].append(os.path.join(root, nome))

    # Filtra apenas grupos com todas as bandas necessárias
    matches: Dict[str, Dict[str, str]] = {}
    incompletos = 0
    for chave, lista in grupos.items():
        bandas = identificar_bandas(lista)
        if BANDAS_NECESSARIAS.issubset(bandas.keys()):
            matches[chave] = bandas
        else:
            incompletos += 1
            faltando = BANDAS_NECESSARIAS - bandas.keys()
            print(f"  ⚠️  {chave}: bandas ausentes {faltando}")

    if incompletos:
        print(f"\n{incompletos} grupo(s) ignorados por bandas incompletas.")

    return matches


# ---------------------------------------------------------------------------
# 2. Funções de cálculo dos índices espectrais
# ---------------------------------------------------------------------------

def _divisao_segura(num: np.ndarray, den: np.ndarray) -> np.ndarray:
    """
    Realiza divisão elemento a elemento, retornando NaN onde denominador é zero.

    Parâmetros:
        num (np.ndarray): Numerador.
        den (np.ndarray): Denominador.

    Retorna:
        np.ndarray: Resultado float32 com NaN onde den == 0.
    """
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.where(den == 0, np.nan, num / den).astype(np.float32)


def calc_ndvi(nir: np.ndarray, red: np.ndarray) -> np.ndarray:
    """
    NDVI = (NIR - RED) / (NIR + RED)

    Índice de vegetação por diferença normalizada. Valores entre -1 e 1;
    vegetação densa produz valores próximos de 1.
    """
    return _divisao_segura(nir - red, nir + red)


def calc_nbr(nir: np.ndarray, swir2: np.ndarray) -> np.ndarray:
    """
    NBR = (NIR - SWIR2) / (NIR + SWIR2)

    Normalized Burn Ratio. Sensível a áreas queimadas; valores negativos
    indicam superfícies queimadas.
    """
    return _divisao_segura(nir - swir2, nir + swir2)


def calc_nbr2(swir: np.ndarray, swir2: np.ndarray) -> np.ndarray:
    """
    NBR2 = (SWIR - SWIR2) / (SWIR + SWIR2)

    Variante do NBR usando duas bandas SWIR. Sensível à umidade do solo
    e cobertura vegetal pós-fogo.
    """
    return _divisao_segura(swir - swir2, swir + swir2)


def calc_ndmi(nir: np.ndarray, swir: np.ndarray) -> np.ndarray:
    """
    NDMI = (NIR - SWIR) / (NIR + SWIR)

    Normalized Difference Moisture Index. Indica o conteúdo hídrico
    da vegetação; valores positivos = maior umidade foliar.
    """
    return _divisao_segura(nir - swir, nir + swir)


def calc_mirbi(swir2: np.ndarray, swir: np.ndarray) -> np.ndarray:
    """
    MIRBI = 10 * SWIR2 - 9.8 * SWIR + 2

    Mid-Infrared Burned Index. Índice linear para detecção de queimadas;
    não requer divisão, portanto sem risco de denominador zero.
    """
    return (10 * swir2 - 9.8 * swir + 2).astype(np.float32)


def calc_evi(nir: np.ndarray, red: np.ndarray, blue: np.ndarray) -> np.ndarray:
    """
    EVI = (NIR - RED) / (NIR + 6*RED - 7.5*BLUE + 1)

    Enhanced Vegetation Index. Menos saturado que o NDVI em áreas de
    alta biomassa e mais resistente a efeitos atmosféricos.
    """
    den = nir + 6 * red - 7.5 * blue + 1
    return _divisao_segura(nir - red, den)


def calc_savi(nir: np.ndarray, red: np.ndarray, L: float = 0.5) -> np.ndarray:
    """
    SAVI = ((NIR - RED) / (NIR + RED + L)) * (1 + L)

    Soil-Adjusted Vegetation Index. O fator L (padrão 0.5) minimiza o
    efeito do solo em áreas de baixa densidade de vegetação.

    Parâmetros adicionais:
        L (float): Fator de correção do solo (padrão: 0.5).
    """
    den = nir + red + L
    return (_divisao_segura(nir - red, den) * (1 + L)).astype(np.float32)


# Catálogo de índices disponíveis para iteração no pipeline principal
INDICES: Dict[str, Callable] = {
    "NDVI":  calc_ndvi,
    "NBR":   calc_nbr,
    "NBR2":  calc_nbr2,
    "NDMI":  calc_ndmi,
    "MIRBI": calc_mirbi,
    "EVI":   calc_evi,
    "SAVI":  calc_savi,
}


# ---------------------------------------------------------------------------
# 3. Leitura e validação de bandas
# ---------------------------------------------------------------------------

def _ler_bandas(
    bandas_caminhos: Dict[str, Optional[str]],
    bandas_necessarias: List[str],
) -> tuple[Dict[str, np.ndarray], Optional[dict]]:
    """
    Lê as bandas necessárias para um índice, verificando alinhamento geométrico.

    A primeira banda lida define o perfil de referência (CRS, transform,
    dimensões). As demais são verificadas contra essa referência.

    Parâmetros:
        bandas_caminhos   (dict): {nome_banda: caminho_arquivo | None}
        bandas_necessarias (list): Nomes das bandas requeridas pelo índice.

    Retorna:
        tuple:
            dict[str, np.ndarray]: Arrays das bandas lidas.
            dict | None: Perfil rasterio da primeira banda, ou None se falha.

    Levanta:
        ValueError: Se dimensões ou transformada de uma banda divergirem
                    da referência.
        FileNotFoundError: Se o caminho de uma banda necessária for None.
    """
    dados: Dict[str, np.ndarray] = {}
    profile = None

    for nome in bandas_necessarias:
        caminho = bandas_caminhos.get(nome)
        if caminho is None:
            raise FileNotFoundError(f"Caminho da banda '{nome}' não encontrado.")

        with rasterio.open(caminho) as src:
            if profile is None:
                # Primeira banda: define referência geométrica
                profile = src.profile.copy()
                dados[nome] = src.read(1).astype(np.float32)
            else:
                # Verifica alinhamento com a referência
                if src.shape != (profile["height"], profile["width"]):
                    raise ValueError(
                        f"Banda '{nome}': dimensões {src.shape} divergem da "
                        f"referência ({profile['height']}, {profile['width']})"
                    )
                if src.transform != profile["transform"]:
                    raise ValueError(
                        f"Banda '{nome}': transformada diverge da referência."
                    )
                dados[nome] = src.read(1).astype(np.float32)

    return dados, profile


# ---------------------------------------------------------------------------
# 4. Processamento principal
# ---------------------------------------------------------------------------

def processar_indice(
    matches_dict: Dict[str, Dict[str, str]],
    nome_indice: str,
    func_calculo: Callable,
    output_base_dir: str,
) -> int:
    """
    Calcula um índice espectral para todos os grupos de data/hora disponíveis.

    Para cada grupo, determina automaticamente quais bandas a função de
    cálculo requer (via inspeção de assinatura), lê apenas essas bandas,
    calcula o índice e exporta o resultado como GeoTIFF.

    Estrutura de saída:
        <output_base_dir>/<nome_indice>/<ano>/<dia>/<hora_HH>/<horaHHMMSS>.tif

    Parâmetros:
        matches_dict   (dict): Retorno de `encontrar_matches_por_data_hora`.
        nome_indice    (str):  Nome do índice (ex.: 'NDVI').
        func_calculo   (callable): Função de cálculo do índice.
        output_base_dir (str): Diretório raiz de saída.

    Retorna:
        int: Número de rasters gerados com sucesso.
    """
    output_base = Path(output_base_dir)

    # Determina as bandas necessárias via inspeção de assinatura (seguro e preciso)
    sig = inspect.signature(func_calculo)
    bandas_necessarias = [
        p for p in sig.parameters
        if p in BAND_MAP   # ignora parâmetros como 'L' (constante do SAVI)
    ]

    sucesso = 0
    iterable = tqdm(matches_dict.items(), desc=f"Calculando {nome_indice}")

    for key, bandas in iterable:
        try:
            # Extrai componentes temporais da chave _sYYYYDDDHHMMSS
            match = _PADRAO_TEMPORAL.search(key)
            if not match:
                print(f"\n  ⚠️  Chave inválida ignorada: {key}")
                continue

            ano, dia, hora = match.group(1), match.group(2), match.group(3)

            # Garante que hora tem pelo menos 2 caracteres para hora_abv
            if len(hora) < 2:
                print(f"\n  ⚠️  Hora inválida na chave {key}: '{hora}'")
                continue
            hora_abv = hora[:2]   # subdiretório de hora (apenas HH)

            # Lê as bandas necessárias com verificação de alinhamento
            dados, profile = _ler_bandas(bandas, bandas_necessarias)

            # Calcula o índice espectral
            resultado = func_calculo(**dados).astype(np.float32)

            # Configura o perfil de saída
            profile.update({
                "dtype":   "float32",
                "count":   1,
                "compress": "lzw",
                "tiled":   True,
                "nodata":  np.nan,
            })

            # Cria diretório de saída e exporta
            output_dir  = output_base / nome_indice / ano / dia / hora_abv
            output_dir.mkdir(parents=True, exist_ok=True)
            output_path = output_dir / f"{hora}.tif"

            with rasterio.open(output_path, "w", **profile) as dst:
                dst.write(resultado, 1)

            sucesso += 1

        except Exception as e:
            print(f"\n  ✘ Erro na chave {key}: {e}")
            continue

    print(f"\n{nome_indice}: {sucesso}/{len(matches_dict)} rasters gerados com sucesso.")
    return sucesso


# ---------------------------------------------------------------------------
# 5. Execução
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    DIRETORIO_BASE = Path("Final")
    OUTPUT_BASE    = Path("Arquivos")

    matches = encontrar_matches_por_data_hora(str(DIRETORIO_BASE))
    print(f"\nGrupos completos encontrados: {len(matches)}\n")

    for nome, func in INDICES.items():
        processar_indice(matches, nome, func, str(OUTPUT_BASE))


Grupos completos encontrados: 12



Calculando NDVI: 100%|██████████| 12/12 [00:00<00:00, 52.46it/s]



NDVI: 12/12 rasters gerados com sucesso.


Calculando NBR: 100%|██████████| 12/12 [00:00<00:00, 59.28it/s]



NBR: 12/12 rasters gerados com sucesso.


...


MIRBI: 12/12 rasters gerados com sucesso.


Calculando EVI: 100%|██████████| 12/12 [00:00<00:00, 37.38it/s]



EVI: 12/12 rasters gerados com sucesso.


Calculando SAVI: 100%|██████████| 12/12 [00:00<00:00, 59.41it/s]


SAVI: 12/12 rasters gerados com sucesso.
